# Experimento N5.1 — Confirmación independiente de Curriculum Learning en DDPG-R2

**Objetivo:** comprobar con cinco semillas nuevas si el currículo de N5 mejora de manera consistente la estabilización estricta del péndulo respecto a DDPG-H02 con entrenamiento directo.

- **Dos tratamientos idénticos a N5:** `H02_DIRECTO` y `N5_CURRICULO`. Se conserva R2, los cuatro estados originales, la física y todos los hiperparámetros H02.
- **Cinco semillas de entrenamiento nuevas:** 901, 902, 903, 904 y 905, emparejadas por tratamiento. Son **10 entrenamientos desde cero**, 160 000 pasos cada uno (**1 600 000 pasos en total**).
- **Currículo inalterado:** 0–40K equilibrio cercano a vertical; 40–80K recuperación; 80–120K mezcla swing-up; 120–160K tarea original. Los cambios se activan solo al reiniciar episodio.
- **Checkpoints:** 40K, 80K, 120K y 160K. **Validación:** 7900–7904; **prueba final reservada:** 9960–9969. Los episodios de prueba no intervienen en selección.
- **Drive obligatorio** en carpeta nueva `N51_VALIDACION_CURRICULUM_DDPG_R2`. Checkpoints y auditorías persistentes; ramas **completadas** se omiten en nuevas sesiones.

**Criterios de confirmación predefinidos:** mejora media pareada ≥10 s, mejora en ≥4 de 5 semillas y no perder políticas exitosas (estabilidad estricta continua media ≥100 s). Son umbrales exploratorios; no constituyen por sí solos una prueba estadística definitiva.

**Precaución:** los 10 episodios por política son mediciones repetidas, no 10 entrenamientos independientes. No comparar N5.1 como si fuera una búsqueda adicional de hiperparámetros.


## 1. Instalar dependencias en Google Colab


In [ ]:
%pip -q install 'stable-baselines3>=2.3,<3' 'gymnasium>=0.29,<2' 'pandas>=2' 'matplotlib>=3'

## 2. Configuración reproducible y guardado persistente

**Obligatorio:** autorizar Google Drive antes de iniciar cualquier entrenamiento. La carpeta `N51_VALIDACION_CURRICULUM_DDPG_R2` contiene los modelos, registros y evaluaciones; no la borres. Puedes cerrar Colab después de una rama completa y reanudar otra sesión ejecutando las celdas iniciales. 


In [ ]:
from pathlib import Path
import os, json, math, hashlib, time, random, zipfile, platform, shutil, tempfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import gymnasium as gym
import stable_baselines3 as sb3
from stable_baselines3 import DDPG
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.callbacks import BaseCallback
from stable_baselines3.common.utils import set_random_seed
from stable_baselines3.common.noise import NormalActionNoise
from google.colab import files, drive

drive.mount('/content/drive')
OUT=Path('/content/drive/MyDrive/N51_VALIDACION_CURRICULUM_DDPG_R2')
OUT.mkdir(parents=True,exist_ok=True)
assert OUT.is_dir() and str(OUT.resolve()).startswith('/content/drive/MyDrive/')

CFG=dict(train_seeds=[901,902,903,904,905], total_steps=160000,
    checkpoints=[40000,80000,120000,160000], learning_starts=1000,
    val_seeds=list(range(7900,7905)), test_seeds=list(range(9960,9970)),
    episode_steps=6000,dt=.02,
    theta_bono_rad=.15, omega_bono_r1_rad_s=.50,
    x_bono_r2_m=.35,v_bono_r2_m_s=.25,bono_r2=3.,
    x_estricto=.20,v_estricto=.10,theta_estricto=.05,omega_estricto=.10)
H02=dict(learning_rate=3e-4,gamma=.995,tau=.010,batch_size=128,sigma=.10,net_arch=[128,128])
TRATAMIENTOS=('H02_DIRECTO','N5_CURRICULO')
ETAPAS={'equilibrio':(0,40000),'recuperacion':(40000,80000),
        'mezcla_swingup':(80000,120000),'tarea_original':(120000,160000)}
assert set(CFG['train_seeds']).isdisjoint(CFG['val_seeds']+CFG['test_seeds'])
assert set(CFG['val_seeds']).isdisjoint(CFG['test_seeds'])
print('Carpeta persistente:',OUT)
print('SB3',sb3.__version__,'Gymnasium',gym.__version__)
print('10 entrenamientos × 160K =',len(TRATAMIENTOS)*len(CFG['train_seeds'])*CFG['total_steps'],'pasos')


## 3. Entorno físico original y recompensa R2

Se utiliza el **mismo archivo del entorno** que N4.1, conservando la dinámica, los parámetros físicos y la recompensa R2. El currículo se implementa únicamente como un envoltorio de reinicios y no modifica `step()`, la observación ni la física.


In [ ]:
from pathlib import Path
Path('envs').mkdir(exist_ok=True)
Path('envs/__init__.py').write_text('',encoding='utf-8')
ORIGINAL_ENV = '"""\nEntorno Gymnasium: péndulo invertido tipo carro–poste con acción continua.\n\nEstado:\n    s_t = [x_t, x_dot_t, theta_t, theta_dot_t]\n\nAcción:\n    a_t = u(t), fuerza continua aplicada al carro.\n\nConvención angular:\n    theta = 0  -> péndulo vertical hacia arriba.\n    theta = pi -> péndulo hacia abajo.\n"""\n\nfrom __future__ import annotations\n\nimport math\nfrom typing import Optional\n\nimport gymnasium as gym\nfrom gymnasium import spaces\nimport numpy as np\n\n\nclass CartPoleContinuousEnv(gym.Env):\n    metadata = {"render_modes": ["human", "rgb_array"], "render_fps": 50}\n\n    def __init__(\n        self,\n        render_mode: Optional[str] = None,\n        max_episode_steps: int = 1000,\n        force_max: float = 10.0,\n        dt: float = 0.02,\n        x_threshold: float = 2.4,\n        noise_std: float = 0.0,\n        viscous_friction_cart: float = 0.0,\n        viscous_friction_pole: float = 0.0,\n        external_impulse: float = 0.0,\n        impulse_step: Optional[int] = None,\n        random_initial_state: bool = True,\n        start_near_downward: bool = True,\n    ):\n        super().__init__()\n\n        self.gravity = 9.81\n        self.masscart = 1.0\n        self.masspole = 0.1\n        self.total_mass = self.masscart + self.masspole\n        self.length = 0.5\n        self.polemass_length = self.masspole * self.length\n\n        self.force_max = float(force_max)\n        self.dt = float(dt)\n        self.x_threshold = float(x_threshold)\n        self.max_episode_steps = int(max_episode_steps)\n\n        self.noise_std = float(noise_std)\n        self.viscous_friction_cart = float(viscous_friction_cart)\n        self.viscous_friction_pole = float(viscous_friction_pole)\n        self.external_impulse = float(external_impulse)\n        self.impulse_step = impulse_step\n\n        self.random_initial_state = random_initial_state\n        self.start_near_downward = start_near_downward\n\n        self.render_mode = render_mode\n        self.screen = None\n        self.clock = None\n        self.screen_width = 800\n        self.screen_height = 500\n\n        self.action_space = spaces.Box(\n            low=np.array([-self.force_max], dtype=np.float32),\n            high=np.array([self.force_max], dtype=np.float32),\n            dtype=np.float32,\n        )\n\n        high = np.array(\n            [\n                self.x_threshold * 2.0,\n                np.finfo(np.float32).max,\n                math.pi,\n                np.finfo(np.float32).max,\n            ],\n            dtype=np.float32,\n        )\n        self.observation_space = spaces.Box(-high, high, dtype=np.float32)\n\n        self.state: Optional[np.ndarray] = None\n        self.steps = 0\n        self.last_u = 0.0\n        self.cumulative_control_effort = 0.0\n        self.cumulative_reward = 0.0\n\n    @staticmethod\n    def angle_normalize(theta: float) -> float:\n        return ((theta + math.pi) % (2.0 * math.pi)) - math.pi\n\n    def _get_obs(self) -> np.ndarray:\n        assert self.state is not None\n        obs = self.state.copy()\n\n        if self.noise_std > 0:\n            noise = self.np_random.normal(0.0, self.noise_std, size=obs.shape)\n            obs = obs + noise.astype(np.float32)\n\n        obs[2] = self.angle_normalize(float(obs[2]))\n        return obs.astype(np.float32)\n\n    def reset(self, seed: Optional[int] = None, options: Optional[dict] = None):\n        super().reset(seed=seed)\n        self.steps = 0\n        self.last_u = 0.0\n        self.cumulative_control_effort = 0.0\n        self.cumulative_reward = 0.0\n\n        if self.random_initial_state:\n            if self.start_near_downward:\n                x = self.np_random.uniform(low=-0.05, high=0.05)\n                x_dot = self.np_random.uniform(low=-0.05, high=0.05)\n                theta = math.pi + self.np_random.uniform(low=-0.10, high=0.10)\n                theta_dot = self.np_random.uniform(low=-0.05, high=0.05)\n            else:\n                x = self.np_random.uniform(low=-0.5, high=0.5)\n                x_dot = self.np_random.uniform(low=-0.5, high=0.5)\n                theta = self.np_random.uniform(low=-math.pi, high=math.pi)\n                theta_dot = self.np_random.uniform(low=-1.0, high=1.0)\n        else:\n            x, x_dot, theta, theta_dot = 0.0, 0.0, math.pi, 0.0\n\n        theta = self.angle_normalize(theta)\n        self.state = np.array([x, x_dot, theta, theta_dot], dtype=np.float32)\n\n        info = {\n            "cumulative_control_effort": self.cumulative_control_effort,\n            "cumulative_reward": self.cumulative_reward,\n        }\n        return self._get_obs(), info\n\n    def step(self, action):\n        assert self.state is not None\n\n        x, x_dot, theta, theta_dot = map(float, self.state)\n        force = float(np.clip(action[0], -self.force_max, self.force_max))\n\n        if self.impulse_step is not None and self.steps == self.impulse_step:\n            force += self.external_impulse\n\n        force -= self.viscous_friction_cart * x_dot\n\n        costheta = math.cos(theta)\n        sintheta = math.sin(theta)\n\n        temp = (\n            force + self.polemass_length * theta_dot**2 * sintheta\n        ) / self.total_mass\n\n        theta_acc = (\n            self.gravity * sintheta\n            - costheta * temp\n            - self.viscous_friction_pole * theta_dot / max(self.polemass_length, 1e-8)\n        ) / (\n            self.length * (4.0 / 3.0 - self.masspole * costheta**2 / self.total_mass)\n        )\n\n        x_acc = temp - self.polemass_length * theta_acc * costheta / self.total_mass\n\n        x_dot = x_dot + self.dt * x_acc\n        x = x + self.dt * x_dot\n        theta_dot = theta_dot + self.dt * theta_acc\n        theta = theta + self.dt * theta_dot\n        theta = self.angle_normalize(theta)\n\n        self.state = np.array([x, x_dot, theta, theta_dot], dtype=np.float32)\n        self.steps += 1\n\n        reward = self._reward(x, x_dot, theta, theta_dot, force)\n\n        terminated = bool(abs(x) > self.x_threshold)\n        truncated = bool(self.steps >= self.max_episode_steps)\n\n        self.last_u = force\n        self.cumulative_control_effort += force**2 * self.dt\n        self.cumulative_reward += reward\n\n        info = {\n            "x": x,\n            "x_dot": x_dot,\n            "theta": theta,\n            "theta_dot": theta_dot,\n            "u": force,\n            "cumulative_control_effort": self.cumulative_control_effort,\n            "cumulative_reward": self.cumulative_reward,\n            "is_upright": abs(theta) < 0.15,\n        }\n\n        if self.render_mode == "human":\n            self.render()\n\n        return self._get_obs(), float(reward), terminated, truncated, info\n\n    def _reward(self, x: float, x_dot: float, theta: float, theta_dot: float, u: float) -> float:\n        """\n        Recompensa mejorada para swing-up y estabilización.\n        """\n\n        reward = 1.0 + math.cos(theta)\n\n        reward += 0.05\n\n        reward -= 0.20 * x**2\n        reward -= 0.01 * x_dot**2\n        reward -= 0.02 * theta_dot**2\n        reward -= 0.001 * u**2\n\n        if abs(theta) < 0.15:\n            reward += 3.0\n\n        if abs(x) > 0.80 * self.x_threshold:\n            reward -= 5.0 * (abs(x) - 0.80 * self.x_threshold)\n\n        if abs(x) > self.x_threshold:\n            reward -= 100.0\n\n        return reward\n\n    def render(self):\n        try:\n            import pygame\n            from pygame import gfxdraw\n        except ImportError as exc:\n            raise ImportError("Para renderizar instala pygame: pip install pygame") from exc\n\n        if self.screen is None:\n            pygame.init()\n            if self.render_mode == "human":\n                self.screen = pygame.display.set_mode((self.screen_width, self.screen_height))\n            else:\n                self.screen = pygame.Surface((self.screen_width, self.screen_height))\n            self.clock = pygame.time.Clock()\n\n        assert self.state is not None\n        x, _, theta, _ = self.state\n\n        world_width = self.x_threshold * 2\n        scale = self.screen_width / world_width\n        cart_y = int(self.screen_height * 0.65)\n        cart_w = 60\n        cart_h = 30\n        pole_len = scale * (2 * self.length)\n\n        cart_x = int(self.screen_width / 2 + x * scale)\n\n        self.screen.fill((255, 255, 255))\n        import pygame\n\n        pygame.draw.line(\n            self.screen, (0, 0, 0),\n            (0, cart_y + cart_h // 2),\n            (self.screen_width, cart_y + cart_h // 2),\n            2\n        )\n\n        cart_rect = pygame.Rect(cart_x - cart_w // 2, cart_y - cart_h // 2, cart_w, cart_h)\n        pygame.draw.rect(self.screen, (30, 144, 255), cart_rect)\n        pygame.draw.rect(self.screen, (0, 0, 0), cart_rect, 2)\n\n        pivot = (cart_x, cart_y - cart_h // 2)\n        pole_x = pivot[0] + pole_len * math.sin(theta)\n        pole_y = pivot[1] - pole_len * math.cos(theta)\n\n        pygame.draw.line(self.screen, (220, 80, 20), pivot, (pole_x, pole_y), 6)\n        gfxdraw.filled_circle(self.screen, int(pole_x), int(pole_y), 10, (220, 80, 20))\n        gfxdraw.aacircle(self.screen, int(pole_x), int(pole_y), 10, (0, 0, 0))\n\n        font = pygame.font.SysFont("Arial", 18)\n        txt = font.render(f"theta={theta:.2f} rad | u={self.last_u:.2f} N", True, (0, 0, 0))\n        self.screen.blit(txt, (20, 20))\n\n        if self.render_mode == "human":\n            pygame.event.pump()\n            pygame.display.flip()\n            self.clock.tick(self.metadata["render_fps"])\n        else:\n            return np.transpose(\n                np.array(pygame.surfarray.pixels3d(self.screen)), axes=(1, 0, 2)\n            )\n\n    def close(self):\n        if self.screen is not None:\n            import pygame\n            pygame.display.quit()\n            pygame.quit()\n            self.screen = None\n            self.clock = None\n'
Path('envs/cartpole_continuous_env.py').write_text(ORIGINAL_ENV,encoding='utf-8')
print('Entorno original preparado; SHA-256:',hashlib.sha256(ORIGINAL_ENV.encode('utf-8')).hexdigest())
from envs.cartpole_continuous_env import CartPoleContinuousEnv

## 4. Currículo de cuatro etapas y pruebas de equivalencia

La etapa solicitada cambia a los pasos 40K, 80K y 120K; **solo se activa en el siguiente `reset()`**, para no alterar un episodio en marcha. Los estados se muestrean con el generador del propio entorno, asociado a su semilla de entrenamiento.

- 1: equilibrio cercano a vertical y centro.
- 2: desviaciones y velocidades moderadas para practicar recuperación.
- 3: mezcla de situaciones cerca del péndulo abajo y orientaciones más amplias.
- 4: exactamente el reinicio original con péndulo abajo.

**Importante:** la política directa no utiliza nunca el currículo, y la validación y la prueba final usan únicamente reinicios originales en ambos tratamientos.


In [ ]:
class CartPoleRewardR1(CartPoleContinuousEnv):
    def _reward(self,x,x_dot,theta,theta_dot,u):
        r=super()._reward(x,x_dot,theta,theta_dot,u)
        if abs(theta)<CFG['theta_bono_rad'] and abs(theta_dot)>=CFG['omega_bono_r1_rad_s']:
            r-=3.0
        return r

class CartPoleRewardR2(CartPoleRewardR1):
    def _reward(self,x,x_dot,theta,theta_dot,u):
        r=super()._reward(x,x_dot,theta,theta_dot,u)
        local=(abs(x)<CFG['x_bono_r2_m'] and abs(x_dot)<CFG['v_bono_r2_m_s']
               and abs(theta)<CFG['theta_bono_rad'] and abs(theta_dot)<CFG['omega_bono_r1_rad_s'])
        return r+(CFG['bono_r2'] if local else 0.0)

class CurriculumReset(gym.Wrapper):
    """Modifica SOLO los estados de reset. Solicitud de nueva etapa se activa en el siguiente episodio."""
    def __init__(self, env, enabled=False):
        super().__init__(env)
        self.enabled=bool(enabled)
        self.requested_stage='equilibrio' if enabled else 'tarea_original'
        self.active_stage=self.requested_stage
        self.transitions=[]
        self.global_training_step=0

    def set_stage(self, stage, step=None):
        assert stage in ETAPAS
        if not self.enabled:
            assert stage=='tarea_original'
        self.requested_stage=stage
        if step is not None: self.global_training_step=int(step)

    def reset(self, *, seed=None, options=None):
        obs,info=self.env.reset(seed=seed,options=options)
        original_stage=self.active_stage
        self.active_stage=self.requested_stage
        if original_stage!=self.active_stage:
            self.transitions.append(dict(requested_at_or_before=self.global_training_step,
                                         activated_stage=self.active_stage))
        if self.enabled and self.active_stage!='tarea_original':
            rng=self.unwrapped.np_random
            stage=self.active_stage
            if stage=='equilibrio':
                x=rng.uniform(-.05,.05);v=rng.uniform(-.03,.03)
                theta=rng.uniform(-.05,.05);omega=rng.uniform(-.05,.05)
            elif stage=='recuperacion':
                x=rng.uniform(-.35,.35);v=rng.uniform(-.25,.25)
                theta=rng.uniform(-.35,.35);omega=rng.uniform(-.60,.60)
            else:
                # 60 % abajo: swing-up, 40 % orientaciones variadas.
                if rng.random()<.6:
                    x=rng.uniform(-.10,.10);v=rng.uniform(-.10,.10)
                    theta=math.pi+rng.uniform(-.20,.20);omega=rng.uniform(-.20,.20)
                else:
                    x=rng.uniform(-.35,.35);v=rng.uniform(-.30,.30)
                    theta=rng.uniform(-math.pi,math.pi);omega=rng.uniform(-1.0,1.0)
            theta=self.unwrapped.angle_normalize(float(theta))
            self.unwrapped.state=np.array([x,v,theta,omega],dtype=np.float32)
            obs=self.unwrapped._get_obs()
        info=dict(info);info['curriculum_stage']=self.active_stage
        return obs,info

    def step(self,action):
        obs,r,term,trunc,info=self.env.step(action)
        info=dict(info);info['curriculum_stage']=self.active_stage
        return obs,r,term,trunc,info

class StateAudit(gym.Wrapper):
    def step(self,action):
        obs,reward,term,trunc,info=self.env.step(action)
        info=dict(info)
        x,v,th,w=map(float,self.unwrapped.state)
        info['local_r2_activado']=int(abs(x)<CFG['x_bono_r2_m'] and abs(v)<CFG['v_bono_r2_m_s']
                                       and abs(th)<CFG['theta_bono_rad'] and abs(w)<CFG['omega_bono_r1_rad_s'])
        info['cerca_vertical']=int(abs(th)<CFG['theta_bono_rad'])
        info['estricta']=int(abs(x)<=CFG['x_estricto'] and abs(v)<=CFG['v_estricto']
                            and abs(th)<=CFG['theta_estricto'] and abs(w)<=CFG['omega_estricto'])
        info['velocidad_carro_estricta']=int(abs(v)<=CFG['v_estricto'])
        return obs,reward,term,trunc,info

def solicitar_etapa(env, etapa, paso=None):
    """Encuentra CurriculumReset bajo StateAudit/Monitor sin depender de Gymnasium forwarding."""
    actual = env
    while actual is not None:
        if isinstance(actual, CurriculumReset):
            actual.set_stage(etapa, paso)
            return actual
        actual = getattr(actual, 'env', None)
    raise RuntimeError('No se encontró CurriculumReset en la cadena de wrappers')

def make_physical_env(steps=6000):
    return CartPoleRewardR2(max_episode_steps=steps,force_max=10.0,dt=0.02,
        x_threshold=2.4,noise_std=0.0,viscous_friction_cart=0.0,
        viscous_friction_pole=0.0,random_initial_state=True,start_near_downward=True)

def make_agent_env(tratamiento,steps=6000,train_audit=False):
    assert tratamiento in TRATAMIENTOS
    env=CurriculumReset(make_physical_env(steps),enabled=(train_audit and tratamiento=='N5_CURRICULO'))
    return StateAudit(env) if train_audit else env

# Igualdad de acciones, observaciones, física, R2 y reinicio habitual en evaluación.
a=make_agent_env('H02_DIRECTO');b=make_agent_env('N5_CURRICULO')
for seed in [901,7900,9960]:
    oa,_=a.reset(seed=seed);ob,_=b.reset(seed=seed)
    assert oa.shape==ob.shape==(4,) and np.array_equal(oa,ob)
    for u in [1.,-2.,3.,0.]:
        oa,ra,ta,ua,_=a.step(np.array([u],dtype=np.float32))
        ob,rb,tb,ub,_=b.step(np.array([u],dtype=np.float32))
        assert np.array_equal(a.unwrapped.state,b.unwrapped.state)
        assert np.array_equal(oa,ob) and np.isclose(ra,rb) and (ta,ua)==(tb,ub)
a.close();b.close()
# Prueba del currículo sin entrenamiento.
e=make_agent_env('N5_CURRICULO',train_audit=True)
for stage in ETAPAS:
    solicitar_etapa(e,stage)
    obs,info=e.reset(seed=901)
    assert obs.shape==(4,) and np.isfinite(obs).all() and info['curriculum_stage']==stage
    if stage=='equilibrio': assert abs(obs[2])<=.05 and abs(obs[0])<=.05
    if stage=='recuperacion': assert abs(obs[2])<=.35 and abs(obs[0])<=.35
    _,reward,_,_,inf=e.step(np.array([0.],dtype=np.float32))
    assert np.isfinite(reward) and inf['curriculum_stage']==stage
e.close()
print('PRUEBAS OK: entorno y recompensa originales iguales; solo cambia distribución de reset durante entrenamiento.')


## 5. Evaluación física común y selección de checkpoints

**Ambas estrategias se validan y prueban desde abajo**; las etapas sencillas del currículo no se emplean en la evaluación. Se prioriza estabilidad continua estricta, luego ocupación, cercanía a vertical, error angular, velocidad del carro y checkpoint tardío.


In [ ]:
def evaluar(modelo,semilla,tratamiento,steps=6000,guardar_traza=False):
    # Mismo entorno físico de prueba; no hay currículo durante la evaluación.
    env=make_agent_env(tratamiento,steps)
    obs,_=env.reset(seed=int(semilla))
    largo=actual=estable=pasos_cerca=cruces=0
    capturas_1s=0; previo_capturado=False; fraccion_local=0
    cumplen={"x":0,"v":0,"theta":0,"omega":0}
    prev_theta=float(env.unwrapped.state[2])
    rmse_acum=cos_acum=esfuerzo=vel_cruce=0.0
    velocidades_cruce=[]; traza=[];termina_riel=False
    for k in range(steps):
        accion,_=modelo.predict(obs,deterministic=True)
        u=float(np.asarray(accion).reshape(-1)[0])
        obs,_,term,trunc,info=env.step(np.array([u],dtype=np.float32))
        x,v,th,w=map(float,env.unwrapped.state)
        dentro=abs(x)<=.20 and abs(v)<=.10 and abs(th)<=.05 and abs(w)<=.10
        cumplen['x']+=int(abs(x)<=.20)
        cumplen['v']+=int(abs(v)<=.10)
        cumplen['theta']+=int(abs(th)<=.05)
        cumplen['omega']+=int(abs(w)<=.10)
        local=(abs(x)<CFG['x_bono_r2_m'] and abs(v)<CFG['v_bono_r2_m_s'] and abs(th)<CFG['theta_bono_rad'] and abs(w)<CFG['omega_bono_r1_rad_s'])
        fraccion_local+=int(local)
        actual=actual+1 if dentro else 0
        if actual==50: capturas_1s+=1
        largo=max(largo,actual); estable+=int(dentro)
        pasos_cerca+=int(abs(th)<.15)
        rmse_acum+=th*th;cos_acum+=math.cos(th);esfuerzo+=u*u*CFG['dt']
        # Cruce próximo a vertical, detección aproximada por cambio de signo y no por wrap de +/-pi
        if (prev_theta*th<=0 and abs(prev_theta)<.6 and abs(th)<.6):
            cruces+=1; velocidades_cruce.append(abs(w))
        prev_theta=th
        if guardar_traza:
            traza.append(dict(t=(k+1)*CFG['dt'],x=x,v=v,theta=th,omega=w,u=u,
                               estricta=int(dentro),equilibrio_local=int(local),cerca_vertical=int(abs(th)<.15)))
        if term or trunc:
            termina_riel=bool(term); break
    n=k+1;env.close()
    resultado=dict(seed=int(semilla),estabilidad_max_s=largo*CFG['dt'],
      ocupacion_pct=100*estable/n,supervivencia_s=n*CFG['dt'],
      rmse_theta_rad=math.sqrt(rmse_acum/n),cos_theta_medio=cos_acum/n,
      esfuerzo_N2s=esfuerzo,esfuerzo_medio_N2=esfuerzo/(n*CFG['dt']),
      fraccion_cerca_vertical=pasos_cerca/n,fin_por_riel=termina_riel,
      fraccion_equilibrio_local_pct=100*fraccion_local/n, capturas_1s=int(capturas_1s),
      **{f'cumple_{nombre}_pct':100*valor/n for nombre,valor in cumplen.items()},
      cruces_vertical=cruces,
      velocidad_angular_cruce_mediana_rad_s=(float(np.median(velocidades_cruce)) if velocidades_cruce else np.nan))
    return (resultado,pd.DataFrame(traza)) if guardar_traza else resultado

def score(row):
    # Score diagnóstico predefinido; no reemplaza la métrica primaria de estabilidad
    return (row['estabilidad_max_s']/(CFG['episode_steps']*CFG['dt'])
            +0.5*row['ocupacion_pct']/100
            +0.25*(row['cos_theta_medio']+1)/2
            -0.2*float(row['fin_por_riel']))
print('Evaluación preparada: métricas físicas originales y observaciones coherentes con el tratamiento.')

## 6. Entrenamiento de las diez políticas con guardado automático en Drive

**No ejecutes la sección 6 si ya están las diez ramas completas.** Cada una se guarda en Drive y se omite en ejecuciones posteriores. Los checkpoints intermedios sirven para evaluación y respaldo, **pero no contienen el replay buffer**: una rama incompleta debe volver a empezar, sin repetir otras ramas completas.

El currículo solicita una etapa nueva al llegar al umbral, pero la activa únicamente en el siguiente `reset`. Para evitar mezclar etapas en un episodio, no se fuerza un reset a mitad del episodio. Se registran las activaciones reales por etapa.


In [ ]:
def guardar_checkpoint_en_drive(modelo,destino):
    destino=Path(destino);destino.parent.mkdir(parents=True,exist_ok=True)
    with tempfile.TemporaryDirectory(prefix='n5_ckpt_') as tmp:
        p=Path(tmp)/'modelo.zip';modelo.save(str(p))
        assert p.stat().st_size>1000
        with zipfile.ZipFile(p) as z:assert z.testzip() is None
        q=destino.with_name(destino.name+'.partial');shutil.copy2(p,q)
        with zipfile.ZipFile(q) as z:assert z.testzip() is None
        assert hashlib.sha256(p.read_bytes()).digest()==hashlib.sha256(q.read_bytes()).digest()
        q.replace(destino)
    assert destino.exists() and destino.stat().st_size>1000
    print('    Guardado Drive:',destino.name,flush=True)

class N5Callback(BaseCallback):
    def __init__(self,folder,training_treatment):
        super().__init__()
        self.folder=Path(folder);self.treatment=training_treatment
        self.counters=[];self.pending='equilibrio' if training_treatment=='N5_CURRICULO' else 'tarea_original'
        self.reset_bucket()
    def reset_bucket(self):
        self.bucket=dict(n=0,local=0,cerca=0,estricta=0,v_estricta=0,stages={})
    def _on_step(self):
        info=self.locals.get('infos',[{}])[0]
        b=self.bucket;b['n']+=1
        for dst,src in [('local','local_r2_activado'),('cerca','cerca_vertical'),
                        ('estricta','estricta'),('v_estricta','velocidad_carro_estricta')]:
            b[dst]+=int(info.get(src,0))
        stage=info.get('curriculum_stage','tarea_original')
        b['stages'][stage]=b['stages'].get(stage,0)+1
        t=int(self.model.num_timesteps)
        if t in CFG['checkpoints']:
            guardar_checkpoint_en_drive(self.model,self.folder/f'modelo_{t}.zip')
            n=b['n'];self.counters.append(dict(checkpoint=t,n_pasos=n,
                r2_local_pct=100*b['local']/n,cerca_vertical_pct=100*b['cerca']/n,
                estricta_pct=100*b['estricta']/n,v_estricta_pct=100*b['v_estricta']/n,
                **{f'etapa_{k}_pasos':b['stages'].get(k,0) for k in ETAPAS}))
            p=self.folder/'auditoria_estados_entrenamiento.csv';q=p.with_name(p.name+'.partial')
            pd.DataFrame(self.counters).to_csv(q,index=False);q.replace(p)
            print(f'  checkpoint {t:,}: cerca={100*b["cerca"]/n:.2f}%, '+
                  f'estabilidad={100*b["estricta"]/n:.2f}%',flush=True)
            self.reset_bucket()
            if self.treatment=='N5_CURRICULO' and t<CFG['total_steps']:
                stage_next={40000:'recuperacion',80000:'mezcla_swingup',120000:'tarea_original'}[t]
                # DummyVecEnv -> Monitor -> StateAudit -> CurriculumReset.
                solicitar_etapa(self.training_env.envs[0],stage_next,t)
                print('  etapa solicitada:',stage_next,'; activa al siguiente reinicio',flush=True)
        return True

def entrenar(seed,tratamiento):
    folder=OUT/f'seed_{seed}'/tratamiento;folder.mkdir(parents=True,exist_ok=True)
    regpath=folder/'registro.json'
    if regpath.exists():
        reg=json.loads(regpath.read_text())
        if (reg.get('completed') and reg.get('configuracion')==H02
            and reg.get('tipo')==tratamiento and reg.get('steps_end')==CFG['total_steps']
            and all((folder/f'modelo_{cp}.zip').exists() for cp in CFG['checkpoints'])
            and (folder/'auditoria_estados_entrenamiento.csv').exists()
            and all((folder/f'modelo_{cp}.zip').stat().st_size > 1000 for cp in CFG['checkpoints'])):
            print('REUTILIZADO:',seed,tratamiento,flush=True);return
    set_random_seed(int(seed))
    env=Monitor(make_agent_env(tratamiento,CFG['episode_steps'],train_audit=True),
                filename=str(folder/'monitor'))
    env.reset(seed=int(seed))
    noise=NormalActionNoise(mean=np.zeros(1),sigma=np.ones(1)*H02['sigma'])
    model=DDPG('MlpPolicy',env,learning_rate=H02['learning_rate'],gamma=H02['gamma'],
        tau=H02['tau'],batch_size=H02['batch_size'],buffer_size=100000,
        learning_starts=CFG['learning_starts'],train_freq=(1,'step'),gradient_steps=1,
        action_noise=noise,policy_kwargs={'net_arch':H02['net_arch'],'n_critics':1},
        seed=int(seed),device='cpu',verbose=0)
    assert model.num_timesteps==0 and model.replay_buffer.size()==0
    print('ENTRENANDO:',seed,tratamiento,flush=True)
    t0=time.perf_counter()
    try:
        model.learn(total_timesteps=CFG['total_steps'],reset_num_timesteps=True,
                    callback=N5Callback(folder,tratamiento),progress_bar=False)
        assert model.num_timesteps==CFG['total_steps']
        assert all((folder/f'modelo_{cp}.zip').exists() for cp in CFG['checkpoints'])
    finally:env.close()
    reg=dict(completed=True,semilla=int(seed),tipo=tratamiento,
             configuracion=H02,steps_start=0,steps_end=CFG['total_steps'],
             learning_starts=CFG['learning_starts'],minutes=round((time.perf_counter()-t0)/60,3))
    q=regpath.with_name(regpath.name+'.partial')
    q.write_text(json.dumps(reg,indent=2,ensure_ascii=False),encoding='utf-8');q.replace(regpath)
    print('COMPLETADO:',seed,tratamiento,flush=True)

print('Estado de Drive:')
for seed in CFG['train_seeds']:
    for treatment in TRATAMIENTOS:
        p=OUT/f'seed_{seed}'/treatment
        print(seed,treatment,[cp for cp in CFG['checkpoints'] if (p/f'modelo_{cp}.zip').exists()])
for seed in CFG['train_seeds']:
    for treatment in TRATAMIENTOS:
        entrenar(seed,treatment)
print('DIEZ ENTRENAMIENTOS COMPLETADOS.')


## 7. Validación de checkpoints con condiciones finales originales

Si Colab se desconectó, ejecuta de nuevo las celdas 1–5 para restaurar funciones y salta **la sección 6**. Esta validación exige los 40 checkpoints completos en Drive; de lo contrario, identifica cuáles faltan antes de seguir.


In [ ]:
esperados=[OUT/f'seed_{seed}'/tratamiento/f'modelo_{cp}.zip'
           for seed in CFG['train_seeds'] for tratamiento in TRATAMIENTOS
           for cp in CFG['checkpoints']]
faltantes=[str(p) for p in esperados if not p.exists()]
print(f'Checkpoints persistentes en Drive: {len(esperados)-len(faltantes)}/{len(esperados)}')
assert not faltantes, 'Faltan checkpoints en Drive:\n'+'\n'.join(faltantes)
rows=[]
for seed in CFG['train_seeds']:
    for etiqueta in TRATAMIENTOS:
        for cp in CFG['checkpoints']:
            path=OUT/f'seed_{seed}'/etiqueta/f'modelo_{cp}.zip'
            assert path.exists(),f'Falta {path}'
            model=DDPG.load(str(path),device='cpu')
            assert model.num_timesteps==cp and model.batch_size==H02['batch_size']
            for evalseed in CFG['val_seeds']:
                res=evaluar(model,evalseed,etiqueta,CFG['episode_steps'])
                rows.append(dict(train_seed=seed,tratamiento=etiqueta,checkpoint=cp,
                                 eval_seed=evalseed,**{k:v for k,v in res.items() if k!='seed'}))
valid=pd.DataFrame(rows)
assert len(valid)==5*2*4*5
valid.to_csv(OUT/'validacion_checkpoints.csv',index=False)
metrics_val=['estabilidad_max_s','ocupacion_pct','fraccion_cerca_vertical',
             'rmse_theta_rad','cumple_v_pct','cumple_x_pct']
summary=(valid.groupby(['train_seed','tratamiento','checkpoint'],as_index=False)[metrics_val].mean())
summary.to_csv(OUT/'resumen_validacion.csv',index=False)
selected=(summary.sort_values(
    ['train_seed','tratamiento','estabilidad_max_s','ocupacion_pct',
     'fraccion_cerca_vertical','rmse_theta_rad','cumple_v_pct','checkpoint'],
    ascending=[True,True,False,False,False,True,False,False],kind='stable')
    .drop_duplicates(['train_seed','tratamiento']).copy())
assert len(selected)==10
selected.to_csv(OUT/'checkpoints_seleccionados_SOLO_VALIDACION.csv',index=False)
(OUT/'seleccion_sha256.txt').write_text(hashlib.sha256(
    (OUT/'checkpoints_seleccionados_SOLO_VALIDACION.csv').read_bytes()).hexdigest()+'\n')
print(selected.to_string(index=False))
print('Selección congelada; NO utilizar datos de pruebas para modificarla.')

## 8. Evaluación final reservada — semillas 9960–9969

No utilizar estos episodios para seleccionar checkpoints. La unidad inferencial es la semilla independiente (3), **no** cada episodio. El éxito requiere aprendizaje reproducible y desempeño en condiciones originales de *swing-up*.


In [ ]:
rows=[]
for row in selected.itertuples(index=False):
    path=OUT/f'seed_{row.train_seed}'/row.tratamiento/f'modelo_{row.checkpoint}.zip'
    model=DDPG.load(str(path),device='cpu')
    for evalseed in CFG['test_seeds']:
        guardar=(evalseed==CFG['test_seeds'][0])
        resultado=evaluar(model,evalseed,row.tratamiento,CFG['episode_steps'],guardar_traza=guardar)
        if guardar:
            vals,traza=resultado
            traza.to_csv(OUT/f'trayectoria_{row.tratamiento}_train_{row.train_seed}_eval_{evalseed}.csv',index=False)
        else:vals=resultado
        rows.append(dict(train_seed=int(row.train_seed),tratamiento=row.tratamiento,
             checkpoint=int(row.checkpoint),eval_seed=evalseed,
             **{k:v for k,v in vals.items() if k!='seed'}))
test=pd.DataFrame(rows)
assert len(test)==100 and set(test.eval_seed)==set(CFG['test_seeds'])
test.to_csv(OUT/'prueba_final_9960_9969.csv',index=False)
metrics=['estabilidad_max_s','ocupacion_pct','rmse_theta_rad','cumple_x_pct',
         'cumple_v_pct','cumple_theta_pct','cumple_omega_pct',
         'supervivencia_s','esfuerzo_medio_N2','fraccion_cerca_vertical']
by_policy=test.groupby(['train_seed','tratamiento'],as_index=False)[metrics].mean()
by_policy.to_csv(OUT/'resumen_por_politica.csv',index=False)
agg=by_policy.groupby('tratamiento')[metrics].agg(['mean','std'])
agg.columns=['_'.join(x) for x in agg.columns]
agg.to_csv(OUT/'resumen_entre_semillas.csv')
pivot=by_policy.pivot(index='train_seed',columns='tratamiento',values='estabilidad_max_s')
assert set(pivot.columns)==set(TRATAMIENTOS)
deltas=(pivot['N5_CURRICULO']-pivot['H02_DIRECTO']).rename('delta_CURRICULO_menos_DIRECTO_s')
deltas.to_csv(OUT/'diferencias_pareadas_CURRICULO_menos_DIRECTO.csv',header=True)
exitos=by_policy.assign(exito_100s=by_policy.estabilidad_max_s>=100.)
tasas=exitos.groupby('tratamiento')['exito_100s'].sum()
reglas=dict(mejora_media_10s=bool(deltas.mean()>=10),
            mejora_4_de_5_semillas=bool((deltas>0).sum()>=4),
            sin_perder_exitos_100s=bool(tasas['N5_CURRICULO']>=tasas['H02_DIRECTO']))
decision=('CONFIRMACION INDEPENDIENTE FAVORABLE N5.1 (n=5); AUN EXPLORATORIA' if all(reglas.values())
          else 'N5.1: NO SE CONFIRMA MEJORA REPRODUCIBLE')
(OUT/'decision_predefinida.json').write_text(json.dumps(dict(
    reglas=reglas,decision=decision,n_semillas=5,
    delta_media_s=float(deltas.mean()),
    mejoras_por_semilla=int((deltas>0).sum()),
    exitos_100s={k:int(v) for k,v in tasas.items()}),
    ensure_ascii=False,indent=2),encoding='utf-8')
print('Resultados por política:\n',by_policy.round(3).to_string(index=False))
print('Diferencias pareadas CURRICULO - DIRECTO:\n',deltas.round(3).to_string())
print('Resumen delta: media=',round(float(deltas.mean()),3),'mediana=',round(float(deltas.median()),3),'min=',round(float(deltas.min()),3),'max=',round(float(deltas.max()),3))
print('Éxitos ≥100s:',tasas.to_dict())
print('Reglas:',reglas,'→',decision)
print('ADVERTENCIA: 5 semillas independientes de entrenamiento; 50 episodios por tratamiento NO son 50 réplicas.')

## 9. Integridad, gráficas, manifiesto y exportación final

Se comprueba que los 40 checkpoints estén en Drive y que existan 200 episodios de validación y 100 de prueba. El ZIP se crea en el almacenamiento temporal solamente **después** de que todos los resultados estén persistidos en Drive; se puede volver a generar si la descarga falla.


In [ ]:
fig,axs=plt.subplots(1,2,figsize=(12,4.5))
for j,m in enumerate(['estabilidad_max_s','ocupacion_pct']):
    for seed in CFG['train_seeds']:
        part=by_policy[by_policy.train_seed==seed].set_index('tratamiento')
        axs[j].plot(list(TRATAMIENTOS),part.loc[list(TRATAMIENTOS),m].values,
                    marker='o',label=f'Semilla {seed}')
    axs[j].set_title('Estabilidad continua máxima' if j==0 else 'Ocupación estricta')
    axs[j].set_ylabel('segundos' if j==0 else '%')
    axs[j].grid(alpha=.25)
axs[0].legend(fontsize=8);fig.tight_layout()
fig.savefig(OUT/'comparacion_N51.png',dpi=150)
plt.show()
for name,df in [('validacion',valid),('final',test),('por_politica',by_policy)]:
    faltantes=df.isna().sum()
    permitidos={'velocidad_angular_cruce_mediana_rad_s'}
    inesperados=[c for c in faltantes[faltantes>0].index if c not in permitidos]
    assert not inesperados,(name,inesperados)
    if 'velocidad_angular_cruce_mediana_rad_s' in df:
        mask=df['velocidad_angular_cruce_mediana_rad_s'].isna()
        assert (df.loc[mask,'cruces_vertical']==0).all()
    print(name,'filas:',len(df),'faltantes:',faltantes[faltantes>0].to_dict())
audit=[]
for seed in CFG['train_seeds']:
    for et in TRATAMIENTOS:
        p=OUT/f'seed_{seed}'/et/'auditoria_estados_entrenamiento.csv'
        assert p.exists(),f'Falta auditoría: {p}'
        a=pd.read_csv(p)
        assert len(a)==4 and list(a.checkpoint)==CFG['checkpoints']
        assert a.n_pasos.sum()==CFG['total_steps']
        assert a[['r2_local_pct','cerca_vertical_pct','estricta_pct','v_estricta_pct']].notna().all().all()
        a.insert(0,'tratamiento',et);a.insert(0,'train_seed',seed)
        audit.append(a)
audit=pd.concat(audit,ignore_index=True)
audit.to_csv(OUT/'auditoria_estados_entrenamiento_TODOS.csv',index=False)
print('Auditoría durante entrenamiento (último segmento por modelo):')
print(audit[audit.checkpoint==CFG['total_steps']][['train_seed','tratamiento','r2_local_pct',
                                     'cerca_vertical_pct','estricta_pct','v_estricta_pct']]
      .round(4).to_string(index=False))
manifest=dict(experimento='N51_validacion_curriculum_DDPG_R2',
    pregunta='¿El aprendizaje progresivo por condiciones iniciales mejora estabilidad DDPG?' ,
    configuracion=CFG,hiperparametros=H02,tratamientos=TRATAMIENTOS,etapas_curriculo=ETAPAS,
    representacion='Ambas ramas usan [x,v,theta,omega]. N5 altera solo condiciones iniciales de entrenamiento.',
    recompensa_R2='R1 + bono local uniforme +3 (histórica; inalterada)',
    validacion=CFG['val_seeds'],evaluacion_final=CFG['test_seeds'],
    seleccion='media estricta > ocupacion > fraccion_cerca_vertical > menor RMSE angular > cumple_v > checkpoint tardio',
    unidad_inferencial='semilla de entrenamiento (n=5); diez episodios por política son observaciones repetidas',
    pasos_totales=1600000,
    limite_interpretacion='Validación independiente de N5 con cinco semillas nuevas; los criterios exploratorios no prueban superioridad universal',
    entorno_sha256=hashlib.sha256(ORIGINAL_ENV.encode('utf-8')).hexdigest(),
    software=dict(stable_baselines3=sb3.__version__,gymnasium=gym.__version__,python=platform.python_version()))
(OUT/'manifiesto.json').write_text(json.dumps(manifest,indent=2,ensure_ascii=False),encoding='utf-8')
archive=Path('/content/experimento_N51_VALIDACION_CURRICULUM_DDPG_R2_resultados.zip')
with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED,compresslevel=4) as z:
    for p in OUT.rglob('*'):
        if p.is_file():z.write(p,p.as_posix())
print('ZIP:',archive,'MB:',round(archive.stat().st_size/1e6,2))
files.download(str(archive))

## 10. Lectura científica, respaldo y reanudación

- La unidad de comparación es la **semilla de entrenamiento (n=5)**; cada política se prueba con diez episodios reservados. Calcular diferencias pareadas `N5_CURRICULO − H02_DIRECTO`.
- Criterios de confirmación fijados antes de entrenar: **ganancia media ≥10 s**, mejora en **al menos 4 de 5 semillas** y **sin perder** políticas con estabilidad continua media ≥100 s.
- Examinar también ocupación estricta, posición y velocidad del carro, esfuerzo de control, sobrevivencia y variabilidad entre semillas.
- Si la sesión termina tras entrenar, volver a ejecutar instalación, configuración, definiciones de entorno y evaluación; **no hay que entrenar de nuevo las ramas completas**. Las secciones de validación y exportación son independientes.
- Los archivos de checkpoints se guardan de forma inmediata en **Google Drive**. Se puede repetir desde cero una *rama interrumpida*, porque aquí no se serializa la memoria de reproducción (*replay buffer*) necesaria para un reinicio exacto de mitad de entrenamiento.
- La evaluación final **9960–9969** debe permanecer reservada: no modificar la selección de checkpoints después de verla.
